In [ ]:
from pathlib import Path

from matplotlib import pyplot as plt

from mach3sbitools.data_loaders import ParaketDataset
from mach3sbitools.simulator import Simulator

# The Command Line Interface
The main interface you should be using with MaCh3 SBI Tools is in the command line! The main command is
```bash
mach3sbi
```

You can list sub-commands and get help for those with `--help`

## `mach3sbi simulate`
This will generate you N simulations and save them to a feather file! It can also save your prior to external file which is needed during the ML training!
```sh
mkdir observations
mach3sbi simulate -m "my_simulator" -s "MySimulator" -c physics_configs/PhysicsConfig.yaml -o data/my_data.feather -n 300000 -r "prior/my_prior.pkl"
```


In [ ]:
# Let's first load back in your simulator
# First things first, let's use the MaCh3SBITools Simulator instance

# Our config file
physics_config = Path("physics_configs/PhysicsConfig.yaml")

# We also let it know that our cyclical parameter is cyclical!
mach3sbi_simulator = Simulator("my_simulator", "MySimulator", physics_config)

In [ ]:
# Once you've run this in terminal let's check they exist!
output = Path("data/my_data.feather")

if not output.is_file():
    raise FileExistsError("Cannot find the file, make sure the output path is correct!")

# We can open the file with the built in dataloader (it loops over a folder of sims usually)
data = ParaketDataset(
    output.parent, parameter_names=mach3sbi_simulator.prior.prior_data.parameter_names
)

# Now we can simply get our x and theta (for many files you can index it with [i])

# Remember: theta == your systematics, x==your simulations!
theta, x = data[0]

In [ ]:
# Now let's do a quick pair plot of our sims to check they're okay!
from sbi.analysis import pairplot

# Firstly the parameter plot (the names need to be inverted a bit)
pairplot(
    theta, labels=[[n] for n in mach3sbi_simulator.prior.prior_data.parameter_names]
)

plt.show()

## `mach3sbi save_data`
This simply saves your data histogram so you don't need to open it later! Let's save it to `nominal_data.parquet`
```sh
mkdir observations
mach3sbi save_data -m "my_simulator" -s "MySimulator" -c physics_configs/PhysicsConfig.yaml --cyclical_pars cyclical_param  -o observations/observed_data.paraquet
```


In [ ]:
# We can look at the save data with Paraquet
import numpy as np
from pyarrow import parquet

table_file = Path("observations/observed_data.paraquet")
if not table_file.is_file():
    raise FileExistsError(
        f"Please run mach3sbi save_data -m 'my_simulator' -s 'MySimulator' -c physics_configs/PhysicsConfig.yaml --cyclical_pars cyclical_param  -o {table_file!s}"
    )

table = parquet.read_table(table_file)
table_arr = np.array(table["data"])


# We can plot the histogram of the data as well (even if we don't know the bin edges!)
plt.stairs(table_arr)
plt.show()

With our observations saved and your simulations done, we're now ready to train an SBI algorithm!